In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import mat73

In [2]:
ignore_sessions = [
    (33, 14),
    (37, 23),
    (39, 26),
    (41, 30)
]

In [3]:
root_path = Path("/media/Projects/alana/UnitRefine")

consider_rs = pd.read_csv(root_path / "consider_rs.csv", header=None)
pvals_rs = pd.read_csv(root_path / "pvals_rs.csv", header=None)
category_responses = pd.read_csv(root_path / "category_responses.csv")

session_counts = pd.read_csv(root_path / "sessionCherryCounts.csv")

category_responses["session_idx"] = (
    category_responses.groupby("subjid")["sessid"]
    .transform(lambda x: pd.Categorical(
        x, categories=sorted(x.unique()), ordered=True
    ).codes + 1)
)

category_responses["directory_name"] = session_counts.iloc[category_responses["sessid"] - 1]["sessionname"].tolist()

In [4]:
session_counts[["patient_id",]] = (
    session_counts["sessionname"]
    .str.extract(r"^(\d+).", expand=True)
    .astype(int)
)

In [5]:
remove_indices = []

for subjid, sessid in ignore_sessions:
    indices = category_responses.index[
        (category_responses["subjid"] == subjid)
        & (category_responses["sessid"] == sessid)
    ].tolist()

    remove_indices.extend(indices)

print(remove_indices)



[1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1105, 1106, 1107, 1108, 1109, 1110, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1118, 1119, 1120, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 202

In [6]:
# remove mismatching sessions (4 in total)
category_responses = category_responses.drop(index=remove_indices)
pvals_rs = pvals_rs.drop(index=remove_indices)
consider_rs = consider_rs.drop(index=remove_indices)

In [7]:
category_responses["nm_stimulus_responses"] = consider_rs.sum(axis=1)

category_responses["has_response"] = consider_rs.sum(axis=1).astype(bool)

In [8]:
category_responses.to_parquet("response_information.parquet",  engine="pyarrow",
    index=False,)